Análisis Final — Explicabilidad e Impacto en el Negocio


## 1. Importación de Librerías

In [ ]:
import os, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold, KFold
from sklearn.metrics import f1_score, roc_auc_score, mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, mean_absolute_error

sns.set_theme(style="whitegrid")
print("Librerías importadas correctamente.")

## 2. Cargar Dataset Limpio y Modelos

In [ ]:
!wget https://raw.githubusercontent.com/FIJI-1919/TRABAJO_PROGRAMACION_CIENCIA/refs/heads/main/data/dataset_clientes_limpio.csv

In [ ]:
data_limpio = pd.read_csv("dataset_clientes_limpio.csv")

## 3. Re-entrenar Modelos Optimizados
tendría que ser con las carpetas /src, peto tuvimos problemas

In [ ]:
def get_classification_models():
    return {
        "LogisticRegression": Pipeline(steps=[
            ("classifier", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")),
        ]),
        "DecisionTreeClassifier": Pipeline(steps=[
            ("classifier", DecisionTreeClassifier(random_state=42, class_weight="balanced")),
        ]),
        "SVM": Pipeline(steps=[
            ("classifier", SVC(kernel="rbf", probability=True, random_state=42, class_weight="balanced")),
        ]),
    }

def tune_classification_model(pipeline, param_grid, X_train, y_train, method='grid', n_iter=10):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    if method == 'grid':
        buscador = GridSearchCV(pipeline, param_grid, cv=cv, scoring='f1', n_jobs=-1, verbose=0)
    else:
        buscador = RandomizedSearchCV(pipeline, param_grid, n_iter=n_iter,
                                      cv=cv, scoring='f1', n_jobs=-1, random_state=42, verbose=0)
    buscador.fit(X_train, y_train)
    print(f"  Mejor F1 (CV): {buscador.best_score_:.4f} | Params: {buscador.best_params_}")
    return buscador.best_estimator_

# Particiones
SEMILLA = 42
X_clf = df.drop(columns=['abandono'])
y_clf = df['abandono']
feature_names = X_clf.columns.tolist()

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=SEMILLA, stratify=y_clf
)

print("Optimizando modelos (puede tardar unos minutos)...")

mejor_dt_clf = tune_classification_model(
    get_classification_models()['DecisionTreeClassifier'],
    {'classifier__max_depth': [3, 5, 7, 10], 'classifier__min_samples_split': [2, 5, 10]},
    X_train_clf, y_train_clf, method='grid'
)

mejor_lr_clf = tune_classification_model(
    get_classification_models()['LogisticRegression'],
    {'classifier__C': [0.01, 0.1, 1.0, 10.0, 100.0]},
    X_train_clf, y_train_clf, method='grid'
)

mejor_svm = tune_classification_model(
    get_classification_models()['SVM'],
    {'classifier__C': [0.1, 1.0, 10.0], 'classifier__gamma': ['scale', 'auto', 0.01, 0.1]},
    X_train_clf, y_train_clf, method='random', n_iter=8
)

print("\nModelos optimizados listos.")

## 4. Feature Importance — Árbol de Decisión
El árbol calcula la importancia de cada variable según cuánto reduce la impureza de Gini en los nodos.

In [ ]:
arbol = mejor_dt_clf.named_steps['classifier']
importancias = pd.Series(arbol.feature_importances_, index=feature_names)
importancias = importancias.sort_values(ascending=False).head(12)

plt.figure(figsize=(10, 6))
sns.barplot(x=importancias.values, y=importancias.index, palette='viridis')
plt.title('Top 12 Variables más Predictivas del Abandono\n(Árbol de Decisión Optimizado)',
          fontsize=13, fontweight='bold')
plt.xlabel('Importancia Relativa (Reducción de Impureza Gini)')
plt.ylabel('Variable')
plt.tight_layout()
plt.show()

print("\nTop 12 variables:")
for var, imp in importancias.items():
    print(f"  {var:<35}: {imp:.4f}")

## 5. Coeficientes de Regresión Logística
- **Coeficiente positivo (+)** → aumenta la probabilidad de abandono (factor de riesgo).
- **Coeficiente negativo (−)** → disminuye la probabilidad de abandono (factor protector).

In [ ]:
reg_log      = mejor_lr_clf.named_steps['classifier']
coeficientes = pd.Series(reg_log.coef_[0], index=feature_names).sort_values(ascending=False)

top_riesgo     = coeficientes.head(6)
top_proteccion = coeficientes.tail(6)
df_impacto     = pd.concat([top_riesgo, top_proteccion]).sort_values(ascending=False)

plt.figure(figsize=(11, 7))
colores = ['tomato' if v > 0 else 'steelblue' for v in df_impacto.values]
sns.barplot(x=df_impacto.values, y=df_impacto.index, palette=colores)
plt.axvline(x=0, color='black', linestyle='--', lw=1)
plt.title('Factores de Riesgo y Protección — Regresión Logística',
          fontsize=13, fontweight='bold')
plt.xlabel('Coeficiente (rojo = riesgo, azul = protección)')
plt.tight_layout()
plt.show()

## 6. Sistema de Scoring de Riesgo
Asignamos a cada cliente una probabilidad de abandono y lo clasificamos en tres niveles.

In [ ]:
proba = mejor_svm.predict_proba(X_clf)[:, 1]

df_scoring = pd.DataFrame({
    'probabilidad_abandono': proba,
    'abandono_real':         y_clf.values
})

def clasificar_riesgo(p):
    if p >= 0.70:   return 'Alto'
    elif p >= 0.40: return 'Medio'
    else:           return 'Bajo'

df_scoring['nivel_riesgo'] = df_scoring['probabilidad_abandono'].apply(clasificar_riesgo)

print("Distribución por nivel de riesgo:")
print(df_scoring['nivel_riesgo'].value_counts())
print()
print("Porcentaje:")
print(df_scoring['nivel_riesgo'].value_counts(normalize=True).map('{:.1%}'.format))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colores_riesgo = {'Alto': 'tomato', 'Medio': 'orange', 'Bajo': 'mediumseagreen'}
conteo = df_scoring['nivel_riesgo'].value_counts()
axes[0].pie(conteo.values,
            labels=conteo.index,
            colors=[colores_riesgo[k] for k in conteo.index],
            autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 12})
axes[0].set_title('Segmentación por Nivel de Riesgo', fontweight='bold')

axes[1].hist(proba, bins=40, color='steelblue', edgecolor='white')
axes[1].axvline(0.40, color='orange', linestyle='--', lw=2, label='Umbral Medio (0.40)')
axes[1].axvline(0.70, color='tomato',  linestyle='--', lw=2, label='Umbral Alto (0.70)')
axes[1].set_xlabel('Probabilidad de Abandono')
axes[1].set_ylabel('Cantidad de Clientes')
axes[1].set_title('Distribución de Probabilidades', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
print("Top 10 clientes con mayor probabilidad de abandono:")
top10 = df_scoring.sort_values('probabilidad_abandono', ascending=False).head(10).copy()
top10['probabilidad_abandono'] = top10['probabilidad_abandono'].map('{:.1%}'.format)
display(top10)